In [1]:
!pip install transformers datasets accelerate -q

In [4]:
from datasets import Dataset
data = {
    "text": [
        "Artificial intelligence is the new electricity.",
        "Machine learning transforms data into intelligence.",
        "Deep learning enables machines to see and hear.",
        "AI will reshape every industry in the coming decade.",
        "The future belongs to those who understand algorithms.",
        "Neural networks mimic the structure of the brain.",
        "Data is the fuel that powers intelligent systems.",
        "Innovation begins with curiosity and computation.",
        "Automation enhances human productivity.",
        "AI is not magic, it is mathematics at scale."
    ]
}

dataset = Dataset.from_dict(data)
dataset

e:\Ai and Ds\venv_gpu_final\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text'],
    num_rows: 10
})

In [5]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13360.24it/s]


In [4]:
tokenizer.pad_token = tokenizer.eos_token

In [5]:
def tokenize_function(example):
  return tokenizer(
      example['text'],
      truncation=True,
      padding='max_length',
      max_length=64
  )
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 10/10 [00:00<00:00, 2258.28 examples/s]


In [6]:
tokenized_dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 10
})

In [7]:
tokenized_dataset = tokenized_dataset.remove_columns(['text'])

In [8]:
tokenized_dataset.set_format('torch')

In [9]:
from transformers import Trainer, TrainingArguments

In [10]:
training_args = TrainingArguments(
    output_dir='./gpt2-finetuned-ai',
    num_train_epochs=20,
    per_device_train_batch_size=2,
    save_steps=10,
    save_total_limit=2,
    logging_steps=5,
    learning_rate=5e-5,
    weight_decay=-0.01,
    fp16=True
)

In [11]:
trainer = Trainer(
    model=model,
    args= training_args,
    train_dataset = tokenized_dataset
)

In [12]:
from transformers import DataCollatorForLanguageModeling

In [13]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

In [14]:
trainer = Trainer(
    model=model,
    args= training_args,
    train_dataset = tokenized_dataset,
    data_collator = data_collator
)

In [15]:
trainer.train()

c:\Users\abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.139836
10,2.573473
15,1.835927
20,1.469746
25,0.857935
30,0.525765
35,0.318254
40,0.238637
45,0.280391
50,0.479638


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]
c:\Users\abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.10s/it]
c:\Users\abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]
c:\Users\abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

TrainOutput(global_step=100, training_loss=0.7678824764490128, metrics={'train_runtime': 170.7226, 'train_samples_per_second': 1.171, 'train_steps_per_second': 0.586, 'total_flos': 6532300800000.0, 'train_loss': 0.7678824764490128, 'epoch': 20.0})

In [16]:
trainer.save_model('./gpt-finetuned-ai')
tokenizer.save_pretrained('./gpt-finetuned-ai')

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.15s/it]


('./gpt-finetuned-ai\\tokenizer_config.json',
 './gpt-finetuned-ai\\tokenizer.json')

In [17]:
from transformers import pipeline

generator = pipeline(
    'text-generation',
    model='./gpt-finetuned-ai',
    tokenizer='./gpt-finetuned-ai'
)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7545.73it/s]


In [18]:
prompt = 'Artificial intelligence'

output = generator(
    prompt,
    max_length=15,
    num_return_sequences=1,
    temperature=0.2,
    top_k=500,
    top_p=0.2
)

for i, seq in enumerate(output):
    print(f"Generated {i+1}: {seq['generated_text']}\n")

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length', 'num_return_sequences', 'top_k', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=15) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated 1: Artificial intelligence is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electri